In [ ]:
!pip install emcee corner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 2.0 MB/s eta 0:00:00


In [3]:
# ==========================================
# R.O.M. HOLOGRAPHIC DECODER: S0-2 EMPIRICAL DATA
# Zero-Hardcode 6D Extraction with Relational Time-Lock
# ==========================================

import numpy as np
import pandas as pd
import emcee
from scipy.optimize import differential_evolution
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------
C_KMS = 299792.458

# ---------------------------------------------------------
# 0. LOAD EMPIRICAL DATA (S0-2)
# ---------------------------------------------------------
print("Loading empirical dataset (LSR-corrected)...")
df = pd.read_csv('https://raw.githubusercontent.com/AntonRize/WILL/refs/heads/main/DATA/S0-2_DataS1_full.csv')
df.columns = df.columns.str.strip()

t_obs = df['MJD'].values
vz_obs = df['RV_km_s'].values
sigma_vz = df['sigma_km_s'].values

Z_obs = 1.0 + (vz_obs / C_KMS)
sigma_Z = sigma_vz / C_KMS

# ---------------------------------------------------------
# 1. R.O.M. EXACT GEOMETRY ENGINE
# ---------------------------------------------------------
def get_phase_unwrapped(t, t_peri, P, e):
    M_unwrapped = (2 * np.pi / P) * (t - t_peri)
    orbit_count = np.floor(M_unwrapped / (2 * np.pi))
    M_wrapped = M_unwrapped % (2 * np.pi)

    E = M_wrapped.copy()
    for _ in range(30):
        f = E - e * np.sin(E) - M_wrapped
        dE = f / (1 - e * np.cos(E))
        E -= dE
        if np.max(np.abs(dE)) < 1e-10:
            break

    o_wrapped = 2 * np.arctan2(np.sqrt(1 + e) * np.sin(E / 2), np.sqrt(1 - e) * np.cos(E / 2))
    return (o_wrapped % (2 * np.pi)) + orbit_count * 2 * np.pi

def get_romer_delayed_phase(t_obs, K_kms, e, omega, P_days, T_peri):
    o_app_unwrapped = get_phase_unwrapped(t_obs, T_peri, P_days, e)
    o_app = o_app_unwrapped % (2 * np.pi)

    a_proj_km = (K_kms * P_days * 86400 * np.sqrt(1 - e**2)) / (2 * np.pi)
    r_proj = a_proj_km * (1 - e**2) / (1 + e * np.cos(o_app))
    z_km = r_proj * np.sin(o_app + omega)

    t_emit = t_obs - (z_km / (C_KMS * 86400))
    return get_phase_unwrapped(t_emit, T_peri, P_days, e)

def generate_z_raw_dynamic(o_unwrapped, beta, i_inc, beta_z0, e, omega_0):
    tau_sq = (1.0 - beta**2) * (1.0 - 2.0 * beta**2)
    tau_Y_sq = 1.0 - tau_sq

    omega_dynamic = omega_0 + (tau_Y_sq / (1.0 - e**2)) * o_unwrapped
    o = o_unwrapped % (2 * np.pi)

    beta_int = beta / np.sqrt(1 - e**2)
    K = beta_int * np.sin(i_inc)
    beta_los = K * (np.cos(o + omega_dynamic) + e * np.cos(omega_dynamic))

    beta_o_sq = (beta**2) * (1 + e**2 + 2 * e * np.cos(o)) / (1 - e**2)
    kappa_o_sq = 2 * (beta**2) * (1 + e * np.cos(o)) / (1 - e**2)

    if np.any(beta_o_sq >= 1.0) or np.any(kappa_o_sq >= 1.0):
        return np.full_like(o_unwrapped, np.nan)

    Z_sys = (1 - beta_o_sq)**(-0.5) * (1 - kappa_o_sq)**(-0.5)
    return Z_sys * (1 + beta_los) * (1 + beta_z0)

# ---------------------------------------------------------
# 2. STAGE 1: RELATIONAL CHRONOLOGICAL SCOUT (7D)
# ---------------------------------------------------------
# Purpose: Find exact macroscopic timeline (P, T_peri) using full relational physics.
# We discard the beta/i outputs from this stage as they are 1D degenerate.
def scout_objective(params):
    beta, i_inc, vz0_c, e, omega, P, t_peri = params
    try:
        K_kms = (beta / np.sqrt(1 - e**2)) * np.sin(i_inc) * C_KMS
        o_unwrapped = get_romer_delayed_phase(t_obs, K_kms, e, omega, P, t_peri)
        Z_model = generate_z_raw_dynamic(o_unwrapped, beta, i_inc, vz0_c, e, omega)
        if np.any(np.isnan(Z_model)): return 1e10
        return np.sum(((Z_obs - Z_model) / sigma_Z)**2)
    except:
        return 1e10

bounds_scout = [
    (0.003, 0.015), (0.0, np.pi/2), (-100/C_KMS, 100/C_KMS),
    (0.85, 0.95), (0.0, 2*np.pi),
    (15.5 * 365.25, 16.5 * 365.25),   # P
    (58240.0, 58270.0)                # T_peri (Broad 30-day window)
]

print("\nSTAGE 1: Relational Time-Scout (Zero-Hardcode Phase Lock)...")
res_scout = differential_evolution(scout_objective, bounds_scout, strategy='best1bin', maxiter=1000, popsize=30, tol=1e-3, seed=42)

P_SCOUT = res_scout.x[5]
T_PERI_SCOUT = res_scout.x[6]
print(f"-> Macroscopic Timeline Locked: P = {P_SCOUT/365.25:.4f} yrs, Base T_peri = {T_PERI_SCOUT:.3f} MJD")

# ---------------------------------------------------------
# 3. STAGE 2: 6D MCMC BAYESIAN MAPPING (Micro-Relaxation)
# ---------------------------------------------------------
# Period is locked. T_peri floats in a tight +/- 1.5 day window to find the exact hour.
def log_prior(theta):
    beta, i_inc, vz0_c, e, omega, t_peri = theta

    if not (0.003 < beta < 0.015): return -np.inf
    if not (0.0 < i_inc < np.pi/2): return -np.inf
    if not (0.85 < e < 0.95): return -np.inf
    if not (0.0 < omega < 2 * np.pi): return -np.inf

    # Micro-window for T_peri to prevent timeline drift
    if not (T_PERI_SCOUT - 1.5 < t_peri < T_PERI_SCOUT + 1.5): return -np.inf

    vz0_kms = vz0_c * C_KMS
    if not (-50.0 < vz0_kms < 10.0): return -np.inf
    lp_vz0 = -0.5 * ((vz0_kms - (-20.0)) / 10.0)**2

    return lp_vz0

def log_likelihood(theta):
    beta, i_inc, vz0_c, e, omega, t_peri = theta
    try:
        K_kms = (beta / np.sqrt(1 - e**2)) * np.sin(i_inc) * C_KMS
        o_unwrapped = get_romer_delayed_phase(t_obs, K_kms, e, omega, P_SCOUT, t_peri)

        Z_model = generate_z_raw_dynamic(o_unwrapped, beta, i_inc, vz0_c, e, omega)
        if np.any(np.isnan(Z_model)): return -np.inf

        return -0.5 * np.sum(((Z_obs - Z_model) / sigma_Z)**2)
    except:
        return -np.inf

def log_probability(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp): return -np.inf
    return lp + log_likelihood(theta)

print("\nSTAGE 2: MCMC Bayesian Mapping (6D Holographic Recovery)...")
nwalkers = 40
ndim = 6

initial_pos = np.array([0.006, np.radians(45.0), -20.0/C_KMS, 0.885, np.radians(66.0), T_PERI_SCOUT])
pos = initial_pos + 1e-4 * np.random.randn(nwalkers, ndim)

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability)
sampler.run_mcmc(pos, 2500, progress=True)

flat_samples = sampler.get_chain(discard=800, thin=5, flat=True)
med_beta, med_i, med_vz0, med_e, med_omega, med_tperi = np.median(flat_samples, axis=0)

# ---------------------------------------------------------
# 4. PHYSICAL CALCULATIONS & OUTPUT
# ---------------------------------------------------------
i_deg = np.degrees(med_i)
omega_deg = np.degrees(med_omega) % 360
vz0_kms = med_vz0 * C_KMS

delta_phi_will_opt = (2 * np.pi * (3 * med_beta**2 - 2 * med_beta**4)) / (1 - med_e**2)
precession_deg_per_orbit = np.degrees(delta_phi_will_opt)

T_sec = P_SCOUT * 24 * 3600
Rs = T_sec * C_KMS * (med_beta**3) / np.pi
a = Rs / (2 * (med_beta**2))
m_rec = (Rs * 1000 * C_KMS**2 * 1e6) / (2 * 6.6743e-11) / 1.98847e30

print(f"\n{'='*65}")
print("=== R.O.M. ZERO-HARDCODE 6D DECODER (S0-2) ===")
print(f"{'='*65}")
print(f"Period (P):               {P_SCOUT/365.25:.4f} yrs (Auto-Locked)")
print(f"Time of Periapsis (T_p):  {med_tperi:.3f} MJD (Micro-Refined)")
print(f"Eccentricity (e):         {med_e:.5f}")
print(f"Arg of Periapsis (w0):    {omega_deg:.2f} deg")
print(f"Internal Precession:      {precession_deg_per_orbit:.3f} deg/orbit")
print("-" * 65)
print(f"Global Kin. Proj (beta):  {med_beta:.6f}")
print(f"Inclination (i_extracted):{i_deg:.2f} deg")
print(f"Background Drift (v_z0):  {vz0_kms:.2f} km/s")
print("-" * 65)
print(f"Schwarzschild Radius:     {Rs:.2f} km")
print(f"Semi-major axis (a):      {a:.2e} km")
print(f"Calculated Mass (M_sun):  {m_rec:,.2f}")
print(f"{'='*65}")

Loading empirical dataset (LSR-corrected)...

STAGE 1: Relational Time-Scout (Zero-Hardcode Phase Lock)...
-> Macroscopic Timeline Locked: P = 16.0512 yrs, Base T_peri = 58257.807 MJD

STAGE 2: MCMC Bayesian Mapping (6D Holographic Recovery)...


100%|██████████| 2500/2500 [00:53<00:00, 47.03it/s]


=== R.O.M. ZERO-HARDCODE 6D DECODER (S0-2) ===
Period (P):               16.0512 yrs (Auto-Locked)
Time of Periapsis (T_p):  58257.807 MJD (Micro-Refined)
Eccentricity (e):         0.88499
Arg of Periapsis (w0):    66.42 deg
Internal Precession:      0.190 deg/orbit
-----------------------------------------------------------------
Global Kin. Proj (beta):  0.006171
Inclination (i_extracted):46.24 deg
Background Drift (v_z0):  -20.69 km/s
-----------------------------------------------------------------
Schwarzschild Radius:     11357453.17 km
Semi-major axis (a):      1.49e+11 km
Calculated Mass (M_sun):  3,845,630.89
